# Titanic KNN Regression

Train a K-Nearest Neighbors regressor to predict ticket `Fare` from passenger features. This notebook mirrors the classification notebook but uses `KNeighborsRegressor` and saves artifacts for the Streamlit app.

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import pickle
import os

In [4]:
# Load data
data_path = '../data/titanic.csv'  # notebook is in notebooks/
# Use python engine and skip malformed lines so the CSV can be read robustly
df = pd.read_csv(data_path, engine='python', on_bad_lines='skip')
# Normalize column names (handle inconsistent headers)
col_map = {}
for c in df.columns:
    lc = c.strip().lower()
    if lc == 'sibsp':
        col_map[c] = 'SibSp'
    elif lc == 'parch':
        col_map[c] = 'Parch'
    elif lc == 'pclass':
        col_map[c] = 'Pclass'
    elif lc == 'age':
        col_map[c] = 'Age'
    elif lc == 'fare':
        col_map[c] = 'Fare'
    elif lc == 'sex':
        col_map[c] = 'Sex'
    elif lc == 'embarked':
        col_map[c] = 'Embarked'
if col_map:
    df.rename(columns=col_map, inplace=True)
df = df.loc[:, df.columns.str.strip()]
df.head()

,Passengerid,Age,Fare,Sex,SibSp,zero,zero.1,zero.2,zero.3,zero.4,...,zero.12,zero.13,zero.14,Pclass,zero.15,zero.16,Embarked,zero.17,zero.18,2urvived
0,1,22.0,7.2500,0,1,0,0,0,0,0,...,0,0,0,3,0,0,2,0,0,0
1,2,38.0,71.2833,1,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
2,3,26.0,7.9250,1,0,0,0,0,0,0,...,0,0,0,3,0,0,2,0,0,1
3,4,35.0,53.1000,1,1,0,0,0,0,0,...,0,0,0,1,0,0,2,0,0,1
4,28,19.0,263.0000,0,3,0,0,0,0,0,...,0,0,0,1,0,0,2,0,0,0


In [5]:
# Select features and target
features = ['Age', 'Sex', 'SibSp', 'Parch', 'Pclass', 'Embarked']
target = 'Fare'
df = df[features + [target]].copy()
# Basic cleaning
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Fare'] = pd.to_numeric(df['Fare'], errors='coerce')
df['Embarked'] = df['Embarked'].astype(str)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Age       25 non-null     float64
 1   Sex       25 non-null     int64  
 2   SibSp     25 non-null     int64  
 3   Parch     25 non-null     int64  
 4   Pclass    25 non-null     int64  
 5   Embarked  25 non-null     object 
 6   Fare      25 non-null     float64
dtypes: float64(2), int64(4), object(1)
memory usage: 1.5+ KB


In [6]:
# Impute missing values
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].replace('nan', np.nan, inplace=True)
if df['Embarked'].isnull().any():
    df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
# Map categorical to numeric codes consistent with the app
sex_map = {'male': 0, 'female': 1}
embarked_map = {'S': 0, 'C': 1, 'Q': 2}
df['Sex'] = df['Sex'].map(lambda v: sex_map.get(str(v).lower(), 0))
df['Embarked'] = df['Embarked'].map(lambda v: embarked_map.get(str(v).upper(), 0))
# Drop rows where target is missing
df = df.dropna(subset=[target])
df.shape

C:\Users\bsais\AppData\Local\Temp\ipykernel_14180\337567353.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
C:\Users\bsais\AppData\Local\Temp\ipykernel_14180\337567353.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

(25, 7)

In [7]:
# Remove extreme outliers in the target using IQR (optional)
Q1 = df[target].quantile(0.25)
Q3 = df[target].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
df = df[(df[target] >= lower) & (df[target] <= upper)]
df.shape

(23, 7)

In [8]:
# Train / test split
X = df[features].copy()
y = df[target].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((18, 6), (5, 6))

In [9]:
# Scale features
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
# Save scaler artifact
os.makedirs('../models', exist_ok=True)
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('Scaler saved to ../models/scaler.pkl')

Scaler saved to ../models/scaler.pkl


In [10]:
# Grid search for KNeighborsRegressor
param_grid = {
    'n_neighbors': [3,5,7,9,11,13,15],
    'weights': ['uniform','distance'],
    'metric': ['euclidean','manhattan','minkowski']
}
knn = KNeighborsRegressor()
gs = GridSearchCV(knn, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
gs.fit(X_train_scaled, y_train)
print('Best params:', gs.best_params_)
best_model = gs.best_estimator_

Best params: {'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'distance'}


c:\Users\bsais\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:1108: UserWarning: One or more of the test scores are non-finite: [-225.93074254 -181.36707663 -260.07855986 -186.10362926 -293.68458089
 -227.21684169 -363.58814587 -246.15853013 -424.17210853 -263.29865808
 -393.3510848  -240.82896636           nan           nan -236.03757642
 -192.5275589  -292.14525785 -206.66589344 -280.91521222 -216.62466812
 -362.75437188 -236.43060725 -431.52794025 -257.27236919 -396.66768457
 -234.01130727           nan           nan -225.93074254 -181.36707663
 -260.07855986 -186.10362926 -293.68458089 -227.21684169 -363.58814587
 -246.15853013 -424.17210853 -263.29865808 -393.3510848  -240.82896636
           nan           nan]
  warnings.warn(


In [11]:
# Evaluate on test set
y_pred = best_model.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f'Test RMSE: {rmse:.4f}')
print(f'Test R2: {r2:.4f}')

Test RMSE: 11.2695
Test R2: -0.4365


In [12]:
# Save model and feature names for the Streamlit app
with open('../models/knn_reg_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open('../models/feature_names.pkl', 'wb') as f:
    pickle.dump(list(X.columns), f)
print('Saved knn_reg_model.pkl and feature_names.pkl into ../models')

Saved knn_reg_model.pkl and feature_names.pkl into ../models


In [13]:
# Quick example: load artifacts and predict for a sample
import pickle
with open('../models/scaler.pkl','rb') as f:
    scaler = pickle.load(f)
with open('../models/knn_reg_model.pkl','rb') as f:
    model = pickle.load(f)
with open('../models/feature_names.pkl','rb') as f:
    feature_names = pickle.load(f)
sample = pd.DataFrame([[30, 0, 0, 0, 3, 0]], columns=feature_names)  # Age,Sex,SibSp,Parch,Pclass,Embarked
sample_scaled = scaler.transform(sample)
print('Predicted Fare:', model.predict(sample_scaled)[0])

Predicted Fare: 7.785
